In [930]:
import numpy as np
import scipy
from scipy.stats import norm, poisson



In [931]:
dane_sz = []
with open("dane_wys_lab2.txt", 'r') as fi:
    for line in fi:
        if line.split():
            line = [float(x) for x in line.split()]
            dane_sz.append(line)


dane=[]
for item in dane_sz:
    for number in item:
        dane.append(number)


In [932]:
sr_p = np.mean(dane) # średnia z próby
std_p = np.std(dane) # odchylenie standardowe z próby

In [933]:
min = np.min(dane)
max = np.max(dane)
n = len(dane)
rozstep = max-min
k_rob = round(n**0.5, 1)
print(min, max, n , k_rob)


129.8 158.2 70 8.4


In [934]:
h = 3.5
x01 = 128

# test zgodnosci chi kwadrat 

Szereg rozdzielczy - zapewnienie warunku liczebności >= 8

In [935]:
def tworz_szereg_8(dane, h, x01):
    max_val = np.max(dane)
    przedzialy = []
    liczebnosci = []
    srodki = []

    x_left = x01
    x_right = x01 + h

    while x_left < max_val:
        # poszerzaj klasę dopóki liczebność < 8
        n = len([x for x in dane if (x > x_left and x <= x_right)])
        while n < 8 and x_right < max_val:
            x_right += h # skleja klasę z porzednią
            n = len([x for x in dane if (x > x_left and x <= x_right)])

        przedzialy.append([x_left, x_right])
        liczebnosci.append(n)
        srodki.append((x_left + x_right) / 2)

        # przejście do kolejnej klasy
        x_left = x_right
        x_right = x_left + h

      # --- Korekta ostatniej klasy ---
    if liczebnosci[-1] < 8 and len(liczebnosci) > 1:
        # połącz ostatnią klasę z poprzednią
        print('poprawiam ostatnią klasę')
        przedzialy[-2][1] = przedzialy[-1][1]  # rozszerz poprzednią klasę
        liczebnosci[-2] += liczebnosci[-1]     # dodaj liczebność
        srodki[-2] = sum(przedzialy[-2]) / 2   # przelicz środek

        # usuń ostatnią klasę
        przedzialy.pop()
        liczebnosci.pop()
        srodki.pop()

    return przedzialy, liczebnosci, srodki

In [936]:
przedzialy, liczebnosci, srodki = tworz_szereg_8(dane, h, x01)
print(przedzialy, liczebnosci)

[[128, 138.5], [138.5, 142.0], [142.0, 145.5], [145.5, 149.0], [149.0, 152.5], [152.5, 159.5]] [10, 11, 10, 17, 13, 9]


Szereg rozdzielczy na podstawie własnych przedziałów

In [937]:
przedzialy_wlasne = [[128, 138.5],[138.5, 142],[142,145.5],[145.5, 149],[149,152.5],[152.5,159.5]]

In [938]:
def tworz_szereg_z_przedzialow(dane, przedzialy):
    liczebnosci = []
    srodki = []
    for item in przedzialy:
        n1 = len([x for x in dane if (x > item[0] and x <= item[1])])
        liczebnosci.append(n1)
        srodki.append((item[0] + item[1])/2)
    return przedzialy, liczebnosci, srodki

Przygotowanie szeregu do testu Chi kwadrat

In [939]:

przedzialy, liczebnosci, srodki = tworz_szereg_8(dane, h, x01)
print(liczebnosci, srodki)

[10, 11, 10, 17, 13, 9] [133.25, 140.25, 143.75, 147.25, 150.75, 156.0]


# test zgodnosci chi kwadrat 
# testowanie zgodności z rozkładem normalnym

In [940]:
# Hipotezy
srednia = 145.6 # tu podaje sie wartość wybraną np. z  komórki M35 Excela
odch_stand = 6.2 # tu podaje sie wartość wybraną np. z  komórki M36 Excela
print("H0: rozkład wysokości jest rozkładem normalnym z parametrami ", srednia,  ', ', odch_stand)
print("H1: tak nie jest")

H0: rozkład wysokości jest rozkładem normalnym z parametrami  145.6 ,  6.2
H1: tak nie jest


In [941]:
# prawdopodobienstwo teoretyczne

f0_lewe = [norm.cdf(prz[0], loc=srednia, scale=odch_stand) for prz in przedzialy]
f0_prawe = [norm.cdf(prz[1], loc=srednia, scale=odch_stand) for prz in przedzialy]
pi = [prawe - lewe for prawe, lewe in zip(f0_prawe, f0_lewe)]

# liczebnośc teoretyczna
npi = [n*p for p in pi]
print(liczebnosci)
print(npi)

#wyznaczenie wartości statystyki testowej
chi2 = [(np-ni)**2/np for np, ni in zip(npi, liczebnosci)]
print(chi2)
testowa = round(np.sum(chi2),3)
print(testowa)

[10, 11, 10, 17, 13, 9]
[np.float64(8.666445038315976), np.float64(10.826804732633304), np.float64(14.897813833360075), np.float64(15.03048447043672), np.float64(11.118664897959082), np.float64(8.427455826528927)]
[np.float64(0.2052016516541186), np.float64(0.0027705866485065343), np.float64(1.6102080892255917), np.float64(0.2580749428816129), np.float64(0.3183315441785641), np.float64(0.038897484285087006)]
2.433


In [942]:
# wyznaczanie wartości krytycznej
alfa = 0.05
r = 2
st_swobody = len(przedzialy) - r - 1
krytyczna = round(scipy.stats.chi2.isf(alfa, st_swobody),3)
print(krytyczna)



7.815


In [943]:
def hipoteza_wnioski(test, kryt):
    if test < kryt:
        print("Nie ma podstaw do odrzucenia H0")
    else:
        print("Odrzucamy H0 na rzecz H1")

In [944]:
hipoteza_wnioski(testowa, krytyczna)

Nie ma podstaw do odrzucenia H0


# test zgodnosci chi kwadrat 
# testowanie zgodności z rozkładem Poissona

In [945]:
# Hipotezy
srednia = 145.6 # tu podaje sie wartość wybraną np. z  komórki M35 Excela
print("H0: rozkład wysokości jest rozkładem Poissona z parametrem ", srednia)
print("H1: tak nie jest")

H0: rozkład wysokości jest rozkładem Poissona z parametrem  145.6
H1: tak nie jest


In [946]:
# prawdopodobienstwo teoretyczne

f0_lewe = [poisson.cdf(prz[0], mu = srednia) for prz in przedzialy]
f0_prawe = [poisson.cdf(prz[1], mu = srednia) for prz in przedzialy]
pi = [prawe - lewe for prawe, lewe in zip(f0_prawe, f0_lewe)]


# liczebnośc teoretyczna
npi = [n*p for p in pi]
print(liczebnosci)
print(npi)

#wyznaczenie wartości statystyki testowej
chi2 = [(np-ni)**2/np for np, ni in zip(npi, liczebnosci)]
print(chi2)
testowa = round(np.sum(chi2),3)
print(testowa)

[10, 11, 10, 17, 13, 9]
[np.float64(14.35963458346051), np.float64(8.57189676172263), np.float64(6.900361150578219), np.float64(9.042879162147825), np.float64(6.161877340398068), np.float64(10.85566144613825)]
[np.float64(1.323600095172099), np.float64(0.6877923871015259), np.float64(1.3923562531273732), np.float64(7.00172709298294), np.float64(7.588583628758283), np.float64(0.3172058579543183)]
18.311


In [947]:
# wyznaczanie wartości krytycznej
alfa = 0.05
r = 1
st_swobody = len(przedzialy) - r - 1
krytyczna = round(scipy.stats.chi2.isf(alfa, st_swobody),3)
print(krytyczna)


9.488


In [948]:
hipoteza_wnioski(testowa, krytyczna)

Odrzucamy H0 na rzecz H1


# test zgodności lambda Kołmogorowa
# testowanie zgodności z rozkładem normalnym

In [949]:
# Hipotezy
# srednia = sr_p
# odch_stand = std_p
srednia = 145.6 # tu podaje sie wartość wybraną 
odch_stand = 6.2  # tu podaje sie wartość wybraną 
print("H0: rozkład wysokości jest rozkładem normalnym z parametrami ", srednia,  ', ',  odch_stand)
print("H1: tak nie jest")

H0: rozkład wysokości jest rozkładem normalnym z parametrami  145.6 ,  6.2
H1: tak nie jest


Tworzenie szeregu z wąskimi klasami

In [950]:
# funkcja tworzenia zwykłego szeregu
def tworz_szereg(dane,  h, x01): #  h - długośc klasy,  x01 - poczatek pierwszej klasy
    max = np.max(dane)
    x_left = x01
    x_right = x01 
    przedzialy = []
    liczebnosci = []
    srodki = []
    while x_right < max:
        x_right = x_left + h
        n1 = len([x for x in dane if (x > x_left and x <= x_right)])
        przedzialy.append([x_left, x_right])
        liczebnosci.append(n1)
        srodki.append ( round( (x_left + x_right)/2, 2))
        x_left = x_right
    return przedzialy, liczebnosci,  srodki


    

In [951]:
def tworz_szereg_waskie_klasy(dane, h, x01, krok, h_min=1e-6): 
    # h - długośc klasy wynikająca ze zwykłego uporzadkowania w szereg
    #h_min - "prawie" zero
  
    # Tworzy szereg rozdzielczy zmniejszając h, dopóki nie powstanie klasa o liczebności zero.
    # COFNIĘCIE O 1 KROK: zwracamy podział z ostatniego h bez pustych klas.
    # krok - zmiana h o wielkośc kroku czyli h-krok
    
    max_val = np.max(dane)

    ostatnie_przedzialy, ostatnie_liczebnosci,  ostatnie_srodki = tworz_szereg(dane,  h, x01)
    ostatnie_h = h

    while h > h_min:
        przedzialy = []
        liczebnosci = []
        srodki = []

        x_left = x01
        x_right = x01 + h

        zero_class = False # czy jest klasa o liczebności 0

        while x_left < max_val:
            n = len([x for x in dane if (x > x_left and x <= x_right)])
            przedzialy.append([x_left, x_right])
            liczebnosci.append(n)
            srodki.append((x_left + x_right) / 2)

            if n == 0:
                zero_class = True

            x_left = x_right
            x_right += h

        if zero_class:
            # ZWRACAMY OSTATNI DOBRY WYNIK
            return ostatnie_przedzialy, ostatnie_liczebnosci, ostatnie_srodki, ostatnie_h

        # zapisujemy aktualny dobry wynik
        ostatnie_przedzialy = przedzialy
        ostatnie_liczebnosci = liczebnosci
        ostatnie_srodki = srodki
        ostatnie_h = h

        # zmniejsz h i próbuj dalej
        h = round((h - krok),2)

    return ostatnie_przedzialy, ostatnie_liczebnosci, ostatnie_srodki, ostatnie_h


In [952]:
# grupujemy dane w wąskie klasy
h = 3.5
x01 = 129.7
n = len(dane)


In [953]:
przedzialy, liczebnosci, srodki, h_nowe = tworz_szereg_waskie_klasy(dane, h, x01, krok = 0.3, h_min=1e-6)
print(liczebnosci, h_nowe)


[1, 1, 1, 1, 6, 4, 4, 9, 3, 7, 6, 9, 8, 4, 1, 3, 2] 1.7


In [954]:
skumulowany = [liczebnosci[0]]
for i in range(1, len(liczebnosci)):
    skumulowany.append(skumulowany[i-1] + liczebnosci[i])
dystr_empiryczna =  [x/n for x in skumulowany]

In [955]:
# dystrybuanta teoretyczna
F0 = [norm.cdf(xi, loc=srednia, scale=odch_stand) for xi in srodki]
roznice = [abs(f0-f) for f0, f in zip(F0, dystr_empiryczna)]
testowa_K = np.sqrt(n) * np.max(roznice)
print(testowa_K)


0.6694136636808204


In [956]:
# wartość krytyczna odczytana z tablicy lambda Kołmogorowa
alfa = 0.05
krytyczna_K = 1.36 # odczytujemy samodzielnie z tablicy podanej w Excelu
hipoteza_wnioski(testowa_K, krytyczna_K)

Nie ma podstaw do odrzucenia H0
